In [ ]:
# 导入必要的库
import collections  # 用于创建默认字典等数据结构
import math  # 数学运算库，用于计算 BLEU 分数等
import torch  # PyTorch 深度学习框架
from torch import nn  # 导入神经网络模块
from d2l import torch as d2l  # 导入 d2l 工具包，提供数据处理、可视化等辅助功能

In [ ]:
class Seq2SeqEncoder(d2l.Encoder):
    """用于序列到序列学习的 RNN 编码器
    
    编码器的作用是将输入序列（如英文句子）编码成一个固定长度的上下文向量（context vector），
    这个向量包含了输入序列的所有信息，供解码器使用。
    """
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0, **kwargs):
        """初始化编码器
        
        参数:
            vocab_size: 词汇表大小，即有多少个不同的词
            embed_size: 词嵌入维度，每个词用多少维的向量表示
            num_hiddens: 隐藏层大小，GRU 隐藏状态的维度
            num_layers: GRU 层数，堆叠多少层 GRU
            dropout: dropout 概率，用于防止过拟合
        """
        super(Seq2SeqEncoder, self).__init__(**kwargs)
        # 嵌入层：将词的索引转换为稠密向量表示
        # 例如：词索引 5 -> [0.2, -0.1, 0.5, ...] (embed_size 维向量)
        self.embedding = nn.Embedding(vocab_size, embed_size)
        # GRU 循环神经网络：处理序列数据
        # 输入维度是 embed_size，输出（隐藏状态）维度是 num_hiddens
        self.rnn = nn.GRU(embed_size, num_hiddens, num_layers,
                          dropout=dropout)

    def forward(self, X, *args):
        """前向传播
        
        参数:
            X: 输入序列，形状为 (batch_size, num_steps)
               batch_size 是批量大小，num_steps 是序列长度
        
        返回:
            output: 所有时间步的输出，形状 (num_steps, batch_size, num_hiddens)
            state: 最终的隐藏状态，形状 (num_layers, batch_size, num_hiddens)
        """
        # X 的形状: (batch_size, num_steps)
        # 例如：[[3, 5, 7, 2], [1, 8, 4, 0]] 表示 2 个句子，每个句子 4 个词
        
        # 将词索引转换为词嵌入向量
        X = self.embedding(X)
        # X 的形状变为: (batch_size, num_steps, embed_size)
        
        # permute 改变维度顺序：将批量维度和序列维度交换
        # 因为 PyTorch 的 GRU 期望输入形状为 (序列长度, 批量大小, 特征维度)
        X = X.permute(1, 0, 2)
        # X 的形状变为: (num_steps, batch_size, embed_size)
        
        # 通过 GRU 处理序列
        output, state = self.rnn(X)
        # output: 每个时间步的输出，形状 (num_steps, batch_size, num_hiddens)
        # state: 最终隐藏状态，形状 (num_layers, batch_size, num_hiddens)
        #        这个状态包含了整个输入序列的信息，将传递给解码器
        return output, state

In [ ]:
# 测试编码器
# 创建一个编码器实例
encoder = Seq2SeqEncoder(vocab_size=10,  # 词汇表大小为 10
                        embed_size=8,     # 每个词用 8 维向量表示
                        num_hiddens=16,   # 隐藏层大小为 16
                        num_layers=2)     # 使用 2 层 GRU

# 创建一个全零的输入张量，模拟输入序列
# 形状 (4, 7) 表示：4 个句子（批量大小），每个句子 7 个词（时间步）
X = torch.zeros((4, 7), dtype=torch.long)

# 将输入传入编码器
output, state = encoder(X)

# 查看输出形状
# output.shape 应该是 torch.Size([7, 4, 16])
#   - 7: 时间步数（序列长度）
#   - 4: 批量大小
#   - 16: 隐藏层大小
# state.shape 应该是 torch.Size([2, 4, 16])
#   - 2: GRU 层数
#   - 4: 批量大小
#   - 16: 隐藏层大小
output.shape, state.shape

In [ ]:
class Seq2SeqDecoder(d2l.Decoder):
    """用于序列到序列学习的 RNN 解码器
    
    解码器的作用是根据编码器生成的上下文向量，逐步生成目标序列（如法文句子）。
    每次生成一个词，当前词的生成会依赖于：
    1. 编码器的上下文信息
    2. 之前生成的词
    """
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0, **kwargs):
        """初始化解码器
        
        参数:
            vocab_size: 目标语言词汇表大小
            embed_size: 词嵌入维度
            num_hiddens: 隐藏层大小
            num_layers: GRU 层数
            dropout: dropout 概率
        """
        super(Seq2SeqDecoder, self).__init__(**kwargs)
        # 嵌入层：将目标语言的词索引转换为向量
        self.embedding = nn.Embedding(vocab_size, embed_size)
        
        # GRU 循环神经网络
        # 注意：输入维度是 embed_size + num_hiddens
        # 因为每一步的输入包括两部分：
        #   1. 当前词的嵌入向量 (embed_size 维)
        #   2. 编码器的上下文向量 (num_hiddens 维)
        self.rnn = nn.GRU(embed_size + num_hiddens, num_hiddens,
                          num_layers, dropout=dropout)
        
        # 全连接层：将隐藏状态映射到词汇表大小
        # 用于预测下一个词的概率分布
        self.dense = nn.Linear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs, *args):
        """初始化解码器的状态
        
        参数:
            enc_outputs: 编码器的输出 (output, state)
        
        返回:
            编码器的最终隐藏状态，作为解码器的初始状态
            这样解码器就能知道输入序列的信息
        """
        return enc_outputs[1]

    def forward(self, X, state):
        """前向传播
        
        参数:
            X: 解码器输入序列，形状 (batch_size, num_steps)
               在训练时，这是目标序列（如法文句子）
            state: 解码器的隐藏状态，来自编码器或上一时间步
        
        返回:
            output: 预测的词概率分布，形状 (batch_size, num_steps, vocab_size)
            state: 更新后的隐藏状态
        """
        # X 的形状: (batch_size, num_steps)
        # 将词索引转换为嵌入向量，并调整维度顺序
        X = self.embedding(X).permute(1, 0, 2)
        # X 的形状变为: (num_steps, batch_size, embed_size)
        
        # 获取编码器的上下文信息
        # state[-1] 是最后一层的隐藏状态，形状 (batch_size, num_hiddens)
        # unsqueeze(0) 添加时间步维度，变为 (1, batch_size, num_hiddens)
        # repeat 复制这个上下文向量，使其在每个时间步都可用
        context = state[-1].unsqueeze(0).repeat(X.shape[0], 1, 1)
        # context 的形状: (num_steps, batch_size, num_hiddens)
        
        # 将当前词的嵌入和上下文向量拼接在一起
        # 这样解码器在生成每个词时，都能"看到"整个输入序列的信息
        X_and_context = torch.cat((X, context), dim=2)
        # X_and_context 的形状: (num_steps, batch_size, embed_size + num_hiddens)
        
        # 通过 GRU 处理
        output, state = self.rnn(X_and_context, state)
        # output 的形状: (num_steps, batch_size, num_hiddens)
        
        # 通过全连接层，将隐藏状态转换为词汇表上的概率分布
        output = self.dense(output).permute(1, 0, 2)
        # output 的形状: (batch_size, num_steps, vocab_size)
        # 对于每个位置，都有 vocab_size 个分数，表示每个词的可能性
        
        # state 的形状: (num_layers, batch_size, num_hiddens)
        return output, state

In [ ]:
# 测试解码器
# 创建一个解码器实例，参数与编码器保持一致
decoder = Seq2SeqDecoder(vocab_size=10, embed_size=8, num_hiddens=16, num_layers=2)

# 将解码器设置为评估模式（不使用 dropout）
decoder.eval()

# 使用编码器的输出初始化解码器的状态
# 这样解码器就获得了输入序列的信息
state = decoder.init_state(encoder(X))

# 测试解码器的前向传播
# X 是输入（这里用同样的数据测试），state 是从编码器获得的状态
output, state = decoder(X, state)

# 查看输出形状
# output.shape 应该是 torch.Size([4, 7, 10])
#   - 4: 批量大小
#   - 7: 时间步数（序列长度）
#   - 10: 词汇表大小（每个位置预测 10 个词中的哪一个）
# state.shape 应该是 torch.Size([2, 4, 16])
#   - 2: GRU 层数
#   - 4: 批量大小
#   - 16: 隐藏层大小
output.shape, state.shape

In [ ]:
def sequence_mask(X, valid_len, value=0):
    """对序列中的无效位置进行掩码
    
    在处理批量数据时，不同序列的长度可能不同。为了批量处理，
    我们通常将短序列填充到相同长度。但在计算损失时，我们不希望
    填充的部分影响结果，所以需要用掩码将填充位置标记出来。
    
    参数:
        X: 输入张量，形状 (batch_size, max_len)
        valid_len: 每个序列的有效长度，形状 (batch_size,)
        value: 用于填充无效位置的值，默认为 0
    
    返回:
        掩码后的张量，无效位置被设置为 value
    
    示例:
        X = [[1, 2, 3], [4, 5, 6]]
        valid_len = [1, 2]
        结果: [[1, 0, 0], [4, 5, 0]]
        # 第一个序列只有第 1 个位置有效，第二个序列前 2 个位置有效
    """
    # 获取序列的最大长度
    maxlen = X.size(1)
    
    # 创建一个掩码矩阵
    # torch.arange(maxlen) 生成 [0, 1, 2, ..., maxlen-1]
    # [None, :] 增加一个维度，变为 [[0, 1, 2, ..., maxlen-1]]
    # valid_len[:, None] 将 valid_len 从 (batch_size,) 变为 (batch_size, 1)
    # 广播比较：每个序列的每个位置是否小于该序列的有效长度
    mask = torch.arange((maxlen), dtype=torch.float32,
                        device=X.device)[None, :] < valid_len[:, None]
    # mask 是布尔张量，True 表示有效位置，False 表示无效位置
    
    # 将所有无效位置（mask 为 False 的位置）设置为 value
    X[~mask] = value  # ~mask 是取反操作
    return X

# 测试 sequence_mask 函数
X = torch.tensor([[1, 2, 3], [4, 5, 6]])
# 第一个序列有效长度为 1，第二个序列有效长度为 2
sequence_mask(X, torch.tensor([1, 2]))

In [ ]:
class MaskedSoftmaxCELoss(nn.CrossEntropyLoss):
    """带掩码的 Softmax 交叉熵损失函数
    
    标准的交叉熵损失会计算所有位置的损失，包括填充位置。
    但填充位置是无意义的，不应该影响模型训练。
    因此，我们需要一个带掩码的损失函数，只计算有效位置的损失。
    """
    def forward(self, pred, label, valid_len):
        """前向传播计算损失
        
        参数:
            pred: 模型预测，形状 (batch_size, num_steps, vocab_size)
                 每个位置是词汇表上的分数分布
            label: 真实标签，形状 (batch_size, num_steps)
                  每个位置是正确词的索引
            valid_len: 每个序列的有效长度，形状 (batch_size,)
        
        返回:
            每个样本的平均损失，形状 (batch_size,)
        """
        # pred 的形状: (batch_size, num_steps, vocab_size)
        # label 的形状: (batch_size, num_steps)
        
        # 创建权重矩阵，初始全为 1
        weights = torch.ones_like(label)
        
        # 使用 sequence_mask 将无效位置的权重设为 0
        # 这样无效位置的损失就不会被计入
        weights = sequence_mask(weights, valid_len)
        # weights 的形状: (batch_size, num_steps)
        # 有效位置为 1，无效位置为 0
        
        # 设置为不进行归约，获取每个位置的损失
        self.reduction = 'none'
        
        # 计算未加权的损失
        # 注意：CrossEntropyLoss 期望输入形状为 (batch_size, vocab_size, num_steps)
        # 所以需要用 permute 调整 pred 的维度顺序
        unweighted_loss = super(MaskedSoftmaxCELoss, self).forward(
            pred.permute(0, 2, 1), label)
        # unweighted_loss 的形状: (batch_size, num_steps)
        # 每个位置都有一个损失值
        
        # 将损失乘以权重，使无效位置的损失为 0
        # 然后对每个序列求平均（只对有效位置求平均）
        weighted_loss = (unweighted_loss * weights).mean(dim=1)
        # weighted_loss 的形状: (batch_size,)
        # 每个样本一个损失值
        
        return weighted_loss

In [ ]:
# 测试带掩码的损失函数
loss = MaskedSoftmaxCELoss()

# 创建测试数据
# pred: 3 个样本，每个样本 4 个时间步，词汇表大小为 10
# 这里用全 1 的张量模拟预测（实际应该是模型输出）
pred = torch.ones(3, 4, 10)

# label: 3 个样本，每个样本 4 个时间步的真实标签
# 这里用全 1 模拟（实际应该是真实的词索引）
label = torch.ones((3, 4), dtype=torch.long)

# valid_len: 3 个样本的有效长度分别为 4, 2, 0
# 第 1 个样本：4 个位置都有效
# 第 2 个样本：只有前 2 个位置有效
# 第 3 个样本：所有位置都无效（全是填充）
valid_len = torch.tensor([4, 2, 0])

# 计算损失
# 结果应该是 3 个值，对应 3 个样本
# 第 3 个样本的损失应该是 0（因为没有有效位置）
loss(pred, label, valid_len)

In [ ]:
def train_seq2seq(net, data_iter, lr, num_epochs, tgt_vocab, device):
    """训练序列到序列模型
    
    参数:
        net: 序列到序列模型（包含编码器和解码器）
        data_iter: 数据迭代器，提供训练批次
        lr: 学习率
        num_epochs: 训练轮数
        tgt_vocab: 目标语言词汇表
        device: 训练设备（CPU 或 GPU）
    """
    def xavier_init_weights(m):
        """Xavier 初始化权重
        
        Xavier 初始化是一种常用的权重初始化方法，
        可以帮助模型更好地训练，避免梯度消失或爆炸。
        
        参数:
            m: 神经网络模块
        """
        # 如果是线性层，使用 Xavier 均匀初始化
        if type(m) == nn.Linear:
            nn.init.xavier_uniform_(m.weight)
        # 如果是 GRU 层，初始化所有权重参数
        if type(m) == nn.GRU:
            for param in m._flat_weights_names:
                if "weight" in param:
                    nn.init.xavier_uniform_(m._parameters[param])

    # 对模型的所有参数应用 Xavier 初始化
    net.apply(xavier_init_weights)
    
    # 将模型移到指定设备（CPU 或 GPU）
    net.to(device)
    
    # 创建 Adam 优化器
    # Adam 是一种自适应学习率的优化算法，通常效果较好
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    
    # 创建带掩码的损失函数
    loss = MaskedSoftmaxCELoss()
    
    # 将模型设置为训练模式
    # 这会启用 dropout 等训练时特有的行为
    net.train()
    
    # 创建动画器，用于可视化训练过程
    animator = d2l.Animator(xlabel='epoch', ylabel='loss',
                     xlim=[10, num_epochs])
    
    # 开始训练循环
    for epoch in range(num_epochs):
        # 创建计时器，用于统计训练速度
        timer = d2l.Timer()
        
        # 创建累加器，用于统计训练损失和处理的词元数量
        # metric[0] 累计损失，metric[1] 累计词元数
        metric = d2l.Accumulator(2)
        
        # 遍历每个训练批次
        for batch in data_iter:
            # 清零梯度
            # 在 PyTorch 中，梯度是累加的，所以每次反向传播前需要清零
            optimizer.zero_grad()
            
            # 将数据移到指定设备
            # X: 源语言序列（如英文）
            # X_valid_len: 源语言序列的有效长度
            # Y: 目标语言序列（如法文）
            # Y_valid_len: 目标语言序列的有效长度
            X, X_valid_len, Y, Y_valid_len = [x.to(device) for x in batch]
            
            # 创建解码器输入
            # '<bos>' 是 "beginning of sequence" 的缩写，表示序列开始
            # 我们需要在目标序列开头添加 <bos> 标记
            bos = torch.tensor([tgt_vocab['<bos>']] * Y.shape[0],
                          device=device).reshape(-1, 1)
            # bos 的形状: (batch_size, 1)
            
            # 构造解码器输入：<bos> + 目标序列的前 n-1 个词
            # 这叫做"教师强制"（Teacher Forcing）
            # 即训练时，解码器的每一步输入是真实的前一个词，而不是模型预测的词
            # 例如：目标序列是 [3, 5, 7, 2]
            #       解码器输入是 [<bos>, 3, 5, 7]
            #       解码器应预测 [3, 5, 7, 2]
            dec_input = torch.cat([bos, Y[:, :-1]], 1)
            
            # 前向传播：将源序列和解码器输入传入模型
            Y_hat, _ = net(X, dec_input, X_valid_len)
            # Y_hat: 模型预测，形状 (batch_size, num_steps, vocab_size)
            
            # 计算损失
            # Y_hat: 预测的词概率分布
            # Y: 真实的目标序列
            # Y_valid_len: 目标序列的有效长度（用于掩码）
            l = loss(Y_hat, Y, Y_valid_len)
            # l 的形状: (batch_size,)
            
            # 反向传播
            # l.sum() 将所有样本的损失求和，得到一个标量
            # 只有标量才能进行反向传播
            l.sum().backward()
            
            # 梯度裁剪
            # 防止梯度爆炸，将梯度的范数限制在 1 以内
            # 这对于 RNN 等循环网络尤其重要
            d2l.grad_clipping(net, 1)
            
            # 计算这个批次处理的词元总数
            # 用于后续计算平均损失和训练速度
            num_tokens = Y_valid_len.sum()
            
            # 更新模型参数
            # 根据计算出的梯度，用优化器更新参数
            optimizer.step()
            
            # 统计损失和词元数
            # 使用 torch.no_grad() 避免在统计时记录梯度
            with torch.no_grad():
                metric.add(l.sum(), num_tokens)
        
        # 每 10 个 epoch 更新一次动画
        if (epoch + 1) % 10 == 0:
            # metric[0] / metric[1] 是平均损失（总损失 / 总词元数）
            animator.add(epoch + 1, (metric[0] / metric[1],))
    
    # 打印训练结果
    # 平均损失、训练速度（词元/秒）、设备信息
    print(f'loss {metric[0] / metric[1]:.3f}, {metric[1] / timer.stop():.1f} '
        f'tokens/sec on {str(device)}')

In [ ]:
# 设置超参数
embed_size, num_hiddens, num_layers, dropout = 32, 32, 2, 0.1
# embed_size=32: 词嵌入维度为 32
# num_hiddens=32: GRU 隐藏层大小为 32
# num_layers=2: 使用 2 层 GRU
# dropout=0.1: dropout 概率为 0.1，用于防止过拟合

batch_size, num_steps = 64, 10
# batch_size=64: 每个批次 64 个样本
# num_steps=10: 每个序列最多 10 个时间步（词）

lr, num_epochs, device = 0.005, 300, d2l.try_gpu()
# lr=0.005: 学习率为 0.005
# num_epochs=300: 训练 300 个 epoch
# device: 尝试使用 GPU，如果没有则使用 CPU

# 加载英法翻译数据集
train_iter, src_vocab, tgt_vocab = d2l.load_data_nmt(batch_size, num_steps)
# train_iter: 训练数据迭代器
# src_vocab: 源语言（英语）词汇表
# tgt_vocab: 目标语言（法语）词汇表

# 创建编码器
# len(src_vocab) 是源语言词汇表的大小
encoder = Seq2SeqEncoder(len(src_vocab), embed_size, num_hiddens, num_layers,
                        dropout)

# 创建解码器
# len(tgt_vocab) 是目标语言词汇表的大小
decoder = Seq2SeqDecoder(len(tgt_vocab), embed_size, num_hiddens, num_layers,
                        dropout)

# 创建完整的编码器-解码器模型
# d2l.EncoderDecoder 将编码器和解码器组合在一起
net = d2l.EncoderDecoder(encoder, decoder)

# 开始训练
# 这将运行 300 个 epoch，可能需要较长时间
train_seq2seq(net, train_iter, lr, num_epochs, tgt_vocab, device)

In [ ]:
def predict_seq2seq(net, src_sentence, src_vocab, tgt_vocab, num_steps,
                    device, save_attention_weights=False):
    """序列到序列模型的预测
    
    这个函数用于将训练好的模型应用于实际翻译任务。
    给定一个源语言句子，模型会逐词生成目标语言的翻译。
    
    参数:
        net: 训练好的序列到序列模型
        src_sentence: 源语言句子（字符串，如 "go ."）
        src_vocab: 源语言词汇表
        tgt_vocab: 目标语言词汇表
        num_steps: 最大生成长度
        device: 预测设备（CPU 或 GPU）
        save_attention_weights: 是否保存注意力权重（用于可视化）
    
    返回:
        output_seq: 生成的目标语言句子（字符串）
        attention_weight_seq: 注意力权重序列（如果 save_attention_weights=True）
    """
    # 在预测时将模型设置为评估模式
    # 这会禁用 dropout 等训练时特有的行为
    net.eval()
    
    # 预处理源语言句子
    # 1. 转换为小写
    # 2. 分词（按空格分割）
    # 3. 将每个词转换为词汇表中的索引
    # 4. 添加序列结束标记 '<eos>'
    src_tokens = src_vocab[src_sentence.lower().split(' ')] + [
        src_vocab['<eos>']]
    # 例如："go ." -> ['go', '.'] -> [词索引1, 词索引2, <eos>索引]
    
    # 记录有效长度（用于编码器）
    enc_valid_len = torch.tensor([len(src_tokens)], device=device)
    
    # 将源序列截断或填充到固定长度 num_steps
    # 如果太长则截断，如果太短则用 '<pad>' 填充
    src_tokens = d2l.truncate_pad(src_tokens, num_steps, src_vocab['<pad>'])
    
    # 添加批量维度
    # 从 (num_steps,) 变为 (1, num_steps)
    # 因为模型期望批量输入，即使我们只预测一个句子
    enc_X = torch.unsqueeze(
        torch.tensor(src_tokens, dtype=torch.long, device=device), dim=0)
    
    # 编码器前向传播
    # 将源语言句子编码为上下文向量
    enc_outputs = net.encoder(enc_X, enc_valid_len)
    
    # 使用编码器的输出初始化解码器状态
    # 解码器将基于这个状态生成翻译
    dec_state = net.decoder.init_state(enc_outputs, enc_valid_len)
    
    # 准备解码器的初始输入
    # 解码器总是从 '<bos>'（序列开始）标记开始生成
    dec_X = torch.unsqueeze(torch.tensor(
        [tgt_vocab['<bos>']], dtype=torch.long, device=device), dim=0)
    # dec_X 的形状: (1, 1) - 一个样本，一个时间步
    
    # 初始化输出序列和注意力权重序列
    output_seq, attention_weight_seq = [], []
    
    # 逐词生成翻译
    # 最多生成 num_steps 个词
    for _ in range(num_steps):
        # 解码器前向传播
        # Y: 这一步的预测（词汇表上的分数分布）
        # dec_state: 更新后的解码器状态
        Y, dec_state = net.decoder(dec_X, dec_state)
        # Y 的形状: (1, 1, vocab_size)
        
        # 选择概率最高的词作为预测结果
        # argmax(dim=2) 在词汇表维度上找最大值的索引
        dec_X = Y.argmax(dim=2)
        # dec_X 的形状: (1, 1) - 预测的词索引
        
        # 提取预测的词索引
        pred = dec_X.squeeze(dim=0).type(torch.int32).item()
        
        # 保存注意力权重（如果需要的话）
        # 这对于理解模型在翻译时关注源句子的哪些部分很有帮助
        if save_attention_weights:
            attention_weight_seq.append(net.decoder.attention_weights)
        
        # 如果预测到序列结束标记 '<eos>'，停止生成
        # 这表示翻译已经完成
        if pred == tgt_vocab['<eos>']:
            break
        
        # 将预测的词添加到输出序列
        output_seq.append(pred)
        
        # 注意：下一次循环时，dec_X（当前预测的词）
        # 将作为解码器的输入，用于生成下一个词
        # 这与训练时的"教师强制"不同
        # 训练时用真实的词，预测时用模型自己生成的词
    
    # 将词索引序列转换回文本
    # tgt_vocab.to_tokens() 将索引转换为词
    # ' '.join() 将词列表连接成句子
    return ' '.join(tgt_vocab.to_tokens(output_seq)), attention_weight_seq

In [ ]:
def bleu(pred_seq, label_seq, k):
    """计算 BLEU 分数
    
    BLEU (Bilingual Evaluation Understudy) 是机器翻译中最常用的评估指标。
    它通过比较预测序列和参考序列之间的 n-gram 匹配来衡量翻译质量。
    BLEU 分数越高（最大为 1），翻译质量越好。
    
    参数:
        pred_seq: 预测的序列（字符串，如 "va !"）
        label_seq: 参考序列（真实翻译，字符串，如 "va !"）
        k: 最大 n-gram 长度（通常使用 k=2 或 k=4）
    
    返回:
        BLEU 分数（0 到 1 之间的浮点数）
    
    BLEU 的计算包括两个部分：
    1. 长度惩罚：如果预测太短，会被惩罚
    2. n-gram 精确度：预测序列中有多少 n-gram 在参考序列中出现
    """
    # 将序列分词
    pred_tokens, label_tokens = pred_seq.split(' '), label_seq.split(' ')
    # 例如："va !" -> ['va', '!']
    
    # 获取预测和参考序列的长度
    len_pred, len_label = len(pred_tokens), len(label_tokens)
    
    # 计算长度惩罚
    # 如果预测比参考短（len_pred < len_label），
    # 则 1 - len_label / len_pred < 0，exp 会产生一个小于 1 的惩罚因子
    # 如果预测比参考长或等长，则惩罚因子为 1（不惩罚）
    score = math.exp(min(0, 1 - len_label / len_pred))
    
    # 计算不同长度的 n-gram 精确度
    # n 从 1 到 k（如 1-gram（单词）、2-gram（词对）等）
    for n in range(1, k + 1):
        # num_matches: 匹配的 n-gram 数量
        num_matches, label_subs = 0, collections.defaultdict(int)
        
        # 统计参考序列中所有 n-gram 的出现次数
        # 例如：参考序列 "i am home" 的 2-gram 有 "i am" 和 "am home"
        for i in range(len_label - n + 1):
            # 提取长度为 n 的子序列
            label_subs[' '.join(label_tokens[i: i + n])] += 1
        # label_subs 是一个字典，键是 n-gram，值是出现次数
        
        # 检查预测序列中的 n-gram 是否在参考序列中
        for i in range(len_pred - n + 1):
            # 提取预测序列中的 n-gram
            pred_ngram = ' '.join(pred_tokens[i: i + n])
            # 如果这个 n-gram 在参考序列中出现过（且还有剩余次数）
            if label_subs[pred_ngram] > 0:
                num_matches += 1  # 匹配数加 1
                label_subs[pred_ngram] -= 1  # 减少剩余次数（防止重复计数）
        
        # 计算当前 n-gram 的精确度，并加入总分
        # len_pred - n + 1 是预测序列中 n-gram 的总数
        # num_matches / (len_pred - n + 1) 是精确度
        # math.pow(0.5, n) 是权重（越长的 n-gram 权重越小）
        score *= math.pow(num_matches / (len_pred - n + 1), math.pow(0.5, n))
    
    return score

In [ ]:
# 测试训练好的模型
# 准备一些英文句子和对应的法文翻译
engs = ['go .', "i lost .", 'he\'s calm .', 'i\'m home .']
fras = ['va !', 'j\'ai perdu .', 'il est calme .', 'je suis chez moi .']

# 对每个英文句子进行翻译，并计算 BLEU 分数
for eng, fra in zip(engs, fras):
    # 使用训练好的模型进行翻译
    translation, attention_weight_seq = predict_seq2seq(
        net, eng, src_vocab, tgt_vocab, num_steps, device)
    # translation: 模型生成的法文翻译
    
    # 打印翻译结果和 BLEU 分数
    # BLEU 分数衡量翻译质量：
    #   - 接近 1.0 表示翻译几乎完美
    #   - 接近 0.0 表示翻译质量很差
    # k=2 表示考虑 1-gram 和 2-gram 的匹配
    print(f'{eng} => {translation}, bleu {bleu(translation, fra, k=2):.3f}')

# 示例输出解释：
# go . => va !, bleu 1.000
#   表示成功将 "go ." 翻译为 "va !"，BLEU 分数为 1.0（完美匹配）
# i lost . => j'ai perdu ., bleu 0.866
#   表示翻译基本正确，但可能有细微差别，BLEU 分数为 0.866